# Order PnL Analysis

Read-only analysis of simulated orders in `kalshi_tennis.db`. Settles each order against market results, breaks down PnL by strategy, edge bucket, and price bucket.

In [ ]:
import sqlite3, json, os
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path(os.getcwd()).resolve().parent
DB_PATH = REPO / 'kalshi_tennis.db'

def connect():
    conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True, timeout=30.0)
    conn.row_factory = sqlite3.Row
    return conn

conn = connect()

## 1. Load Orders + Join Markets/Events

In [ ]:
query = '''
SELECT o.ts, o.match_ticker, o.market_ticker, o.action, o.context,
       o.conv_prob, o.market_price, o.edge_cents, o.suggested_size,
       o.set_number, o.payload,
       m.result AS market_result, m.close_ts, m.settlement_ts,
       e.coverage, e.title
FROM orders o
LEFT JOIN markets m ON o.market_ticker = m.market_ticker
LEFT JOIN events e ON o.match_ticker = e.event_ticker
ORDER BY o.ts
'''
df = pd.read_sql_query(query, conn)
print(f'Total orders: {len(df)}')
df.head(10)

## 2. Settlement Logic

In [ ]:
def settle(row):
    if row['market_result'] == 'yes':
        return (1.0 - row['market_price']) * row['suggested_size']
    elif row['market_result'] == 'no':
        return -row['market_price'] * row['suggested_size']
    else:
        return None  # unresolved

df['pnl'] = df.apply(settle, axis=1)
df['resolved'] = df['pnl'].notna()
df['won'] = (df['pnl'] > 0).astype(int)

resolved = df[df['resolved']].copy()
print(f'Resolved: {len(resolved)} / {len(df)}')
print(f'Unresolved: {len(df) - len(resolved)}')
print(f'Total PnL: {resolved["pnl"].sum():.2f}')
print(f'Hit rate: {resolved["won"].mean():.1%}')

## 3. Per-Strategy Breakdown

In [ ]:
def strategy_name(ctx):
    if 'match_point' in ctx:
        return 'match_point'
    elif 'close_timer' in ctx:
        return 'close_timer'
    return 'other'

resolved['strategy'] = resolved['context'].apply(strategy_name)

for strat, grp in resolved.groupby('strategy'):
    wins = grp['won'].sum()
    total = len(grp)
    gross_win = grp[grp['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(grp[grp['pnl'] < 0]['pnl'].sum())
    profit_factor = gross_win / gross_loss if gross_loss > 0 else float('inf')
    sharpe = grp['pnl'].mean() / grp['pnl'].std() if grp['pnl'].std() > 0 else float('inf')
    cum = grp['pnl'].cumsum()
    max_dd = (cum - cum.cummax()).min()
    print(f'\n=== {strat} ===')
    print(f'  Orders: {total}')
    print(f'  Hit rate: {wins/total:.1%}' if total > 0 else '  No orders')
    print(f'  Total PnL: {grp["pnl"].sum():.2f}')
    print(f'  Avg win: {grp[grp["pnl"]>0]["pnl"].mean():.2f}' if wins > 0 else '  No wins')
    print(f'  Avg loss: {grp[grp["pnl"]<0]["pnl"].mean():.2f}' if (grp['pnl']<0).any() else '  No losses')
    print(f'  Profit factor: {profit_factor:.2f}')
    print(f'  Sharpe-like: {sharpe:.2f}')
    print(f'  Max drawdown: {max_dd:.2f}')

## 4. Edge Bucket Analysis

In [ ]:
bins = [0, 10, 20, 30, 1000]
labels = ['5-10', '11-20', '21-30', '30+']
resolved['edge_bucket'] = pd.cut(resolved['edge_cents'], bins=bins, labels=labels, right=True)

for bucket, grp in resolved.groupby('edge_bucket', observed=False):
    if len(grp) == 0:
        continue
    print(f'{bucket}: n={len(grp)}, hit={grp["won"].mean():.1%}, pnl={grp["pnl"].sum():.2f}')

## 5. Price Bucket Analysis

In [ ]:
pbins = [0, 0.50, 0.70, 0.85, 0.92, 1.01]
plabels = ['<0.50', '0.50-0.70', '0.70-0.85', '0.85-0.92', '0.92+']
resolved['price_bucket'] = pd.cut(resolved['market_price'], bins=pbins, labels=plabels, right=False)

for bucket, grp in resolved.groupby('price_bucket', observed=False):
    if len(grp) == 0:
        continue
    print(f'{bucket}: n={len(grp)}, hit={grp["won"].mean():.1%}, pnl={grp["pnl"].sum():.2f}')

## 6. Time Analysis

In [ ]:
resolved['date'] = pd.to_datetime(resolved['ts'], unit='ms').dt.date
daily = resolved.groupby('date').agg(
    orders=('pnl', 'count'),
    pnl=('pnl', 'sum'),
    hit_rate=('won', 'mean')
).reset_index()
print(daily.to_string(index=False))

cum_pnl = resolved.sort_values('ts')['pnl'].cumsum()
print(f'\nCumulative PnL: {cum_pnl.iloc[-1]:.2f}' if len(cum_pnl) > 0 else 'No data')

## 7. Sanity Checks

In [ ]:
issues = []

# All orders should be buy
non_buy = df[df['action'] != 'buy']
if len(non_buy) > 0:
    issues.append(f'{len(non_buy)} non-buy orders found')

# Match point orders should have conv_prob = 0.97
mp_orders = df[df['context'].str.contains('match_point', na=False)]
bad_conv = mp_orders[abs(mp_orders['conv_prob'] - 0.97) > 0.001]
if len(bad_conv) > 0:
    issues.append(f'{len(bad_conv)} match_point orders with conv_prob != 0.97')

# Close timer orders should have conv_prob = 0.95
ct_orders = df[df['context'].str.contains('close_timer', na=False)]
bad_conv_ct = ct_orders[abs(ct_orders['conv_prob'] - 0.95) > 0.001]
if len(bad_conv_ct) > 0:
    issues.append(f'{len(bad_conv_ct)} close_timer orders with conv_prob != 0.95')

# No duplicate orders for same (match_ticker, set_number, game_number, point_number)
if 'payload' in df.columns:
    def extract_point(row):
        try:
            p = json.loads(row['payload']) if row['payload'] else {}
            return f"{row['match_ticker']}:{row['set_number']}:{p.get('game', '?')}:{p.get('set', '?')}"
        except:
            return f"{row['match_ticker']}:{row['set_number']}:?"
    df['point_key'] = df.apply(extract_point, axis=1)
    dups = df[df['context'].str.contains('match_point', na=False)].duplicated('point_key', keep=False)
    if dups.any():
        issues.append(f'{dups.sum()} duplicate match_point orders (same point_key)')

# Close timer: one order per event
ct_dups = ct_orders.duplicated('match_ticker', keep=False)
if ct_dups.any():
    issues.append(f'{ct_dups.sum()} duplicate close_timer orders (same event)')

if issues:
    print('SANITY CHECK FAILURES:')
    for i in issues:
        print(f'  - {i}')
else:
    print('All sanity checks passed.')

conn.close()